In [1]:
from utility import common_load_parquet_dataset
import pandas as pd
from pathlib import Path
import time

In [2]:
DATA_PATH = Path(
    r"E:\repos\LLM_traffic_query\tests\energy_prediction\dataset\ashrae-energy-prediction\processed_features\ashrae_train_cleaned_plus_manual_features"
)
df = common_load_parquet_dataset(DATA_PATH)
print(df.columns)

Index(['building_id', 'timestamp', 'meter_reading', 'ts_idx', 'pair_id',
       'primary_use', 'square_feet', 'year_built', 'floor_count',
       'timestamp_gmt', 'time_diff_hours', 'air_temperature', 'cloud_coverage',
       'dew_temperature', 'precip_depth_1_hr', 'sea_level_pressure',
       'wind_direction', 'wind_speed', 'is_na_holiday', 'is_eu_holiday',
       'hour_sin', 'hour_cos', 'dayofweek', 'doy_sin', 'doy_cos', 'is_weekend',
       'is_holiday_any', 'is_business_hours', 'CDH_18C', 'HDH_18C',
       'is_hot_24C', 'is_cold_10C', 'dewpoint_depression', 'log_sqft',
       'year_built_clipped', 'site_id', 'meter'],
      dtype='str')


In [3]:
# Old code retained for reference.
# from expert_feature_extractor import extract_expert_features
#
# cache = {}
# df_expert = extract_expert_features(df, cache=cache)

# Revised: load all supplementary ablation groups.
from expert_feature_extractor import extract_expert_features
from supplementary_feature_rank2 import extract_rank2_weather_stats_features
from supplementary_feature_rank25 import extract_rank25_missingness_features

try:
    from supplementary_feature_rank4_revised import extract_rank4_features
except ImportError:
    from supplementary_feature_rank4 import extract_rank4_features

try:
    from supplementary_feature_rank5_revised import extract_rank5_mma_features
except ImportError:
    from supplementary_feature_rank5 import extract_rank5_mma_features

SUPPLEMENTARY_EXCLUDE_BUILDING_ID = True

feature_group_frames = {}
feature_group_caches = {}

cache_expert = {}
feature_group_frames["expert"] = extract_expert_features(df, cache=cache_expert)
feature_group_caches["expert"] = cache_expert

cache_rank2 = {}
feature_group_frames["rank2"] = extract_rank2_weather_stats_features(
    df,
    cache=cache_rank2,
    mode="fit_transform",
    label_col="meter_reading",
    exclude_building_id=SUPPLEMENTARY_EXCLUDE_BUILDING_ID,
)
feature_group_caches["rank2"] = cache_rank2

cache_rank4 = {}
feature_group_frames["rank4"] = extract_rank4_features(
    df,
    cache=cache_rank4,
    mode="fit_transform",
    label_col="meter_reading",
    exclude_building_id=SUPPLEMENTARY_EXCLUDE_BUILDING_ID,
)
feature_group_caches["rank4"] = cache_rank4

cache_rank5 = {}
feature_group_frames["rank5"] = extract_rank5_mma_features(
    df,
    cache=cache_rank5,
    mode="fit_transform",
    label_col="meter_reading",
    exclude_building_id=SUPPLEMENTARY_EXCLUDE_BUILDING_ID,
)
feature_group_caches["rank5"] = cache_rank5

cache_rank25 = {}
feature_group_frames["rank25"] = extract_rank25_missingness_features(
    df,
    cache=cache_rank25,
    mode="fit_transform",
    label_col="meter_reading",
)
feature_group_caches["rank25"] = cache_rank25

for group_name, frame in feature_group_frames.items():
    print(f"{group_name}: shape={frame.shape}")

[rank5]: Skipping building features due to exclude_building_id=True or missing building_id column.
expert: shape=(19544538, 16)
rank2: shape=(19544538, 24)
rank4: shape=(19544538, 62)
rank5: shape=(19544538, 8)
rank25: shape=(19544538, 17)


In [4]:
# Old code retained for reference.
# df_full = pd.concat([df, df_expert], axis=1)

# Revised: concatenate baseline with all supplementary groups for per-feature analysis.
feature_group_order = ["expert", "rank2", "rank4", "rank5", "rank25"]
df_full = pd.concat([df] + [feature_group_frames[g] for g in feature_group_order], axis=1)
print(df_full.shape)

(19544538, 164)


In [5]:
df_full.columns

Index(['building_id', 'timestamp', 'meter_reading', 'ts_idx', 'pair_id',
       'primary_use', 'square_feet', 'year_built', 'floor_count',
       'timestamp_gmt',
       ...
       'r25_site_missing_rate_precip_depth_1_hr',
       'r25_is_missing_sea_level_pressure',
       'r25_site_missing_rate_sea_level_pressure',
       'r25_is_missing_wind_direction', 'r25_site_missing_rate_wind_direction',
       'r25_is_missing_wind_speed', 'r25_site_missing_rate_wind_speed',
       'r25_missing_weather_count', 'r25_missing_weather_frac',
       'r25_site_any_missing_rate'],
      dtype='str', length=164)

In [6]:
# Old code retained for reference.
# target_col = "meter_reading"
# cols_base = df.columns.tolist()
# cols_base.remove(target_col)
# cols_base.remove("timestamp")
# cols_exp = df_expert.columns.tolist()
# cols_full = cols_base + cols_exp
# print("full cols:", cols_full)

target_col = "meter_reading"
cols_base = df.columns.tolist()
if target_col in cols_base:
    cols_base.remove(target_col)
if "timestamp" in cols_base:
    cols_base.remove("timestamp")

feature_group_cols = {name: frame.columns.tolist() for name, frame in feature_group_frames.items()}
cols_exp = feature_group_cols["expert"]
cols_rank2 = feature_group_cols["rank2"]
cols_rank4 = feature_group_cols["rank4"]
cols_rank5 = feature_group_cols["rank5"]
cols_rank25 = feature_group_cols["rank25"]

cols_full = cols_base + cols_exp + cols_rank2 + cols_rank4 + cols_rank5 + cols_rank25

feature_origin_map = {c: "base" for c in cols_base}
for group_name, cols in feature_group_cols.items():
    for c in cols:
        feature_origin_map[c] = group_name

print("n base:", len(cols_base))
for group_name in feature_group_order:
    print(f"n {group_name}:", len(feature_group_cols[group_name]))
print("n full:", len(cols_full))

n base: 35
n expert: 16
n rank2: 24
n rank4: 62
n rank5: 8
n rank25: 17
n full: 162


In [7]:
import numpy as np
import pandas as pd

from patsy import dmatrix, build_design_matrices

from sklearn.dummy import DummyRegressor
from sklearn.feature_selection import mutual_info_regression
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import KFold
from sklearn.preprocessing import OneHotEncoder

In [8]:
TRUE_STRINGS = {"true", "1", "yes", "y", "t"}
FALSE_STRINGS = {"false", "0", "no", "n", "f"}


def _make_onehot_encoder(dtype=np.float32):
    """
    Return a sparse OneHotEncoder.
    New sklearn uses sparse_output=...; older sklearn uses sparse=...
    """
    try:
        return OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=True,
            dtype=dtype,
        )
    except TypeError:
        return OneHotEncoder(
            handle_unknown="ignore",
            sparse=True,
            dtype=dtype,
        )


def normalize_boolean_series(s: pd.Series) -> pd.Series:
    """
    Normalize bool-like values to strings 'True'/'False', keep NaN as NaN.
    """

    def _map(v):
        if pd.isna(v):
            return np.nan
        if isinstance(v, (bool, np.bool_)):
            return "True" if bool(v) else "False"

        txt = str(v).strip().lower()
        if txt in TRUE_STRINGS:
            return "True"
        if txt in FALSE_STRINGS:
            return "False"
        return str(v)

    return s.map(_map)


def infer_feature_type(s: pd.Series) -> str:
    """
    Conservative feature type inference:
      - bool / nullable bool / binary {0,1} -> boolean
      - object/string/category -> categorical
      - other numeric -> numeric
    """
    s_nonnull = s.dropna()
    if s_nonnull.empty:
        return "categorical"

    # Native / nullable boolean dtype
    if pd.api.types.is_bool_dtype(s):
        return "boolean"

    # Numeric columns: only pure binary {0,1} are treated as boolean
    if pd.api.types.is_numeric_dtype(s):
        uniq = pd.unique(s_nonnull)
        if len(uniq) <= 2:
            try:
                uniq_as_float = set(pd.Series(uniq).astype(float).tolist())
                if uniq_as_float.issubset({0.0, 1.0}):
                    return "boolean"
            except Exception:
                pass
        return "numeric"

    # object/string/category columns
    if pd.api.types.is_object_dtype(s) or pd.api.types.is_string_dtype(s) or pd.api.types.is_categorical_dtype(s):
        s_norm = normalize_boolean_series(s_nonnull).dropna().astype(str)
        if not s_norm.empty:
            uniq = set(s_norm.unique().tolist())
            if uniq.issubset({"True", "False"}):
                return "boolean"
        return "categorical"

    # Fallback
    return "categorical"


def prepare_discrete_series_for_mi(s: pd.Series, feature_type: str) -> np.ndarray:
    """
    For boolean/categorical MI:
      - fill NaN with '__MISSING__'
      - factorize to integer codes
    """
    if feature_type == "boolean":
        s = normalize_boolean_series(s)
    s = s.astype("object").where(s.notna(), "__MISSING__").astype(str)
    codes, _ = pd.factorize(s, sort=True)
    return codes.astype(int)


def get_feature_group(col: str) -> str:
    if col in cols_base:
        return "base"
    if col in cols_exp:
        return "exp"
    return "other"


# Revised group resolver for expanded ablations.
# def get_feature_group(col: str):
#     if col in cols_base:
#         return "base"
#     if col in cols_exp:
#         return "exp"
#     return "other"


def get_feature_group(col: str) -> str:
    return feature_origin_map.get(col, "other")

In [9]:
# Old code retained for reference.
# detected_feature_types = {col: infer_feature_type(df_full[col]) for col in [c for c in cols_full if c != target_col]}
#
# detected_feature_types

detected_feature_types = {col: infer_feature_type(df_full[col]) for col in [c for c in cols_full if c != target_col]}

pd.Series(detected_feature_types).groupby(
    pd.Series({k: get_feature_group(k) for k in detected_feature_types})
).size().sort_index()

C:\Users\zpz\AppData\Local\Temp\ipykernel_31880\3730604924.py:73: Pandas4Warning: is_categorical_dtype is deprecated and will be removed in a future version. Use isinstance(dtype, pd.CategoricalDtype) instead
  if pd.api.types.is_object_dtype(s) or pd.api.types.is_string_dtype(s) or pd.api.types.is_categorical_dtype(s):
C:\Users\zpz\AppData\Local\Temp\ipykernel_31880\3730604924.py:73: Pandas4Warning: is_categorical_dtype is deprecated and will be removed in a future version. Use isinstance(dtype, pd.CategoricalDtype) instead
  if pd.api.types.is_object_dtype(s) or pd.api.types.is_string_dtype(s) or pd.api.types.is_categorical_dtype(s):
C:\Users\zpz\AppData\Local\Temp\ipykernel_31880\3730604924.py:73: Pandas4Warning: is_categorical_dtype is deprecated and will be removed in a future version. Use isinstance(dtype, pd.CategoricalDtype) instead
  if pd.api.types.is_object_dtype(s) or pd.api.types.is_string_dtype(s) or pd.api.types.is_categorical_dtype(s):


base      35
expert    16
rank2     24
rank25    17
rank4     62
rank5      8
dtype: int64

In [10]:
def mutual_info_single_feature(x: pd.Series, y: pd.Series, feature_type: str, random_state: int = 42) -> float:
    """
    Per-feature MI against numeric target.
    NaN handling:
      - numeric: median imputation
      - boolean/categorical: '__MISSING__' category
    """
    mask = y.notna()
    x = x.loc[mask].reset_index(drop=True)
    y = pd.to_numeric(y.loc[mask], errors="coerce").reset_index(drop=True)

    valid_y = y.notna()
    x = x.loc[valid_y].reset_index(drop=True)
    y = y.loc[valid_y].to_numpy()

    if len(y) == 0:
        return np.nan

    if feature_type == "numeric":
        x_num = pd.to_numeric(x, errors="coerce")

        # constant or all-missing -> MI = 0
        if x_num.notna().sum() == 0 or x_num.nunique(dropna=True) <= 1:
            return 0.0

        fill_value = x_num.median()
        x_num = x_num.fillna(fill_value).to_numpy().reshape(-1, 1)

        mi = mutual_info_regression(x_num, y, discrete_features=False, random_state=random_state)[0]
        return float(mi)

    # boolean / categorical
    x_disc = prepare_discrete_series_for_mi(x, feature_type)
    if pd.Series(x_disc).nunique(dropna=False) <= 1:
        return 0.0

    mi = mutual_info_regression(x_disc.reshape(-1, 1), y, discrete_features=True, random_state=random_state)[0]
    return float(mi)

In [11]:
def predict_numeric_probe_train_test(x_train: pd.Series, y_train: pd.Series, x_test: pd.Series, spline_df: int = 4):
    """
    Numeric feature probe:
      - natural cubic spline with df=4 if enough unique values
      - linear fallback if too few unique values
      - mean fallback if constant / all-missing
    Includes a missingness indicator.
    """
    x_train = pd.to_numeric(x_train, errors="coerce")
    x_test = pd.to_numeric(x_test, errors="coerce")
    y_train = pd.to_numeric(y_train, errors="coerce")

    if len(y_train) == 0:
        return np.array([], dtype=np.float32), "mean_fallback"

    if x_train.notna().sum() == 0:
        model = DummyRegressor(strategy="mean")
        model.fit(np.zeros((len(y_train), 1), dtype=np.float32), y_train)
        pred = model.predict(np.zeros((len(x_test), 1), dtype=np.float32))
        return pred.astype(np.float32, copy=False), "mean_fallback"

    n_unique = x_train.nunique(dropna=True)
    fill_value = x_train.median()

    x_train_imp = x_train.fillna(fill_value)
    x_test_imp = x_test.fillna(fill_value)

    miss_train = x_train.isna().astype(np.float32).to_numpy().reshape(-1, 1)
    miss_test = x_test.isna().astype(np.float32).to_numpy().reshape(-1, 1)

    if n_unique <= 1:
        model = DummyRegressor(strategy="mean")
        model.fit(np.zeros((len(y_train), 1), dtype=np.float32), y_train)
        pred = model.predict(np.zeros((len(x_test), 1), dtype=np.float32))
        return pred.astype(np.float32, copy=False), "mean_fallback"

    if n_unique >= spline_df:
        spline_train = dmatrix(f"cr(x, df={spline_df}) - 1", {"x": x_train_imp.to_numpy()}, return_type="dataframe")
        design_info = spline_train.design_info
        spline_test = build_design_matrices([design_info], {"x": x_test_imp.to_numpy()})[0]

        X_train = np.hstack([np.asarray(spline_train, dtype=np.float32), miss_train]).astype(np.float32, copy=False)
        X_test = np.hstack([np.asarray(spline_test, dtype=np.float32), miss_test]).astype(np.float32, copy=False)

        model = LinearRegression()
        model.fit(X_train, y_train)
        pred = model.predict(X_test)
        return pred.astype(np.float32, copy=False), f"natural_cubic_spline_df{spline_df}"

    X_train = np.hstack([x_train_imp.to_numpy(dtype=np.float32).reshape(-1, 1), miss_train]).astype(
        np.float32, copy=False
    )
    X_test = np.hstack([x_test_imp.to_numpy(dtype=np.float32).reshape(-1, 1), miss_test]).astype(np.float32, copy=False)

    model = LinearRegression()
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    return pred.astype(np.float32, copy=False), "linear_fallback"


def predict_discrete_probe_train_test(
    x_train: pd.Series, y_train: pd.Series, x_test: pd.Series, feature_type: str, ridge_alpha: float = 1.0
):
    """
    Boolean/categorical feature probe:
      - fill NaN as '__MISSING__'
      - sparse one-hot encode
      - sparse-capable ridge regression
    """
    if feature_type == "boolean":
        x_train = normalize_boolean_series(x_train)
        x_test = normalize_boolean_series(x_test)

    x_train = x_train.astype("object").where(x_train.notna(), "__MISSING__").astype(str)
    x_test = x_test.astype("object").where(x_test.notna(), "__MISSING__").astype(str)

    enc = _make_onehot_encoder(dtype=np.float32)

    X_train = enc.fit_transform(x_train.to_frame())  # sparse CSR
    X_test = enc.transform(x_test.to_frame())  # sparse CSR

    if X_train.shape[1] <= 1:
        model = DummyRegressor(strategy="mean")
        model.fit(np.zeros((len(y_train), 1), dtype=np.float32), y_train)
        pred = model.predict(np.zeros((len(x_test), 1), dtype=np.float32))
        return pred.astype(np.float32, copy=False), "mean_fallback"

    model = Ridge(alpha=ridge_alpha, solver="lsqr")
    model.fit(X_train, y_train.to_numpy(dtype=np.float32))
    pred = model.predict(X_test)

    if feature_type == "boolean":
        return np.asarray(pred, dtype=np.float32), "boolean_factor_ridge_sparse"
    return np.asarray(pred, dtype=np.float32), "categorical_factor_ridge_sparse"


def cross_validated_univariate_probe(
    x: pd.Series,
    y: pd.Series,
    feature_type: str,
    n_splits: int = 3,
    random_state: int = 42,
    spline_df: int = 4,
    ridge_alpha: float = 1.0,
    feature_name: str = None,
    verbose: int = 1,
):
    """
    Cross-validated per-feature probe score.
    Returns out-of-fold R^2 and RMSE.
    """
    mask = y.notna()
    x = x.loc[mask].reset_index(drop=True)
    y = pd.to_numeric(y.loc[mask], errors="coerce").reset_index(drop=True)

    valid_y = y.notna()
    x = x.loc[valid_y].reset_index(drop=True)
    y = y.loc[valid_y].reset_index(drop=True)

    n = len(y)
    fname = feature_name if feature_name is not None else "<unknown>"

    if n < 2:
        return {
            "probe_cv_r2": np.nan,
            "probe_cv_rmse": np.nan,
            "probe_model": "not_enough_rows",
            "n_rows_used": n,
        }

    n_splits_eff = min(n_splits, n)
    if n_splits_eff < 2:
        return {
            "probe_cv_r2": np.nan,
            "probe_cv_rmse": np.nan,
            "probe_model": "not_enough_rows",
            "n_rows_used": n,
        }

    cv = KFold(n_splits=n_splits_eff, shuffle=True, random_state=random_state)
    oof_pred = np.full(n, np.nan, dtype=np.float32)
    model_names = []

    probe_t0 = time.perf_counter()

    for fold_idx, (train_idx, test_idx) in enumerate(cv.split(np.arange(n)), start=1):
        fold_t0 = time.perf_counter()

        # if verbose:
        #     print(
        #         f"      fold {fold_idx}/{n_splits_eff} | "
        #         f"feature={fname} | type={feature_type} | "
        #         f"train={len(train_idx):,} test={len(test_idx):,}"
        #     )

        x_train = x.iloc[train_idx]
        y_train = y.iloc[train_idx]
        x_test = x.iloc[test_idx]

        if feature_type == "numeric":
            pred, model_name = predict_numeric_probe_train_test(
                x_train=x_train, y_train=y_train, x_test=x_test, spline_df=spline_df
            )
        else:
            pred, model_name = predict_discrete_probe_train_test(
                x_train=x_train, y_train=y_train, x_test=x_test, feature_type=feature_type, ridge_alpha=ridge_alpha
            )

        oof_pred[test_idx] = pred
        model_names.append(model_name)

        # if verbose:
        #     print(f"        fold done in {time.perf_counter() - fold_t0:.1f}s | " f"model={model_name}")

    probe_r2 = r2_score(y, oof_pred)
    probe_rmse = np.sqrt(mean_squared_error(y, oof_pred))
    probe_model = pd.Series(model_names).mode().iloc[0]

    if verbose:
        print(
            f"      probe finished for {fname} in {time.perf_counter() - probe_t0:.1f}s | "
            f"R2={probe_r2:.6f} | RMSE={probe_rmse:.6f}"
        )

    return {
        "probe_cv_r2": float(probe_r2),
        "probe_cv_rmse": float(probe_rmse),
        "probe_model": probe_model,
        "n_rows_used": int(n),
    }

In [12]:
# Old code retained for reference.
# df_full["building_id"] = df_full["building_id"].astype("category")
# df_full["pair_id"] = df_full["pair_id"].astype("category")
# df_full["ts_idx"] = df_full["ts_idx"].astype("category")  # only if it is an index/code, not ordered numeric meaning

for cat_col in ["building_id", "pair_id", "ts_idx"]:
    if cat_col in df_full.columns:
        df_full[cat_col] = df_full[cat_col].astype("category")

In [13]:
# Features to score: everything except the target
feature_cols = [c for c in cols_full if c != target_col]

# Strongly recommended for 20M rows
sample_size_per_feature = 2000000  # set None to use all rows
cv_splits_per_feature = 3
random_state = 42

# Shared valid target rows
y_full = pd.to_numeric(df_full[target_col], errors="coerce")
valid_idx = y_full.index[y_full.notna()]

# Shared sample across all features for fair comparison and much faster runtime
if sample_size_per_feature is not None and len(valid_idx) > sample_size_per_feature:
    sampled_idx = (
        pd.Series(valid_idx).sample(n=sample_size_per_feature, random_state=random_state).sort_values().to_numpy()
    )
else:
    sampled_idx = valid_idx.to_numpy()

print(f"Using {len(sampled_idx):,} rows for per-feature MI/probe scoring.")

y_all = pd.to_numeric(df_full.loc[sampled_idx, target_col], errors="coerce").reset_index(drop=True)

rows = []
total_t0 = time.perf_counter()

for i, col in enumerate(feature_cols, start=1):
    feat_t0 = time.perf_counter()

    s = df_full.loc[sampled_idx, col].reset_index(drop=True)
    feature_type = infer_feature_type(s)

    print(
        f"[{i}/{len(feature_cols)}] feature={col} | "
        f"type={feature_type} | "
        f"missing={s.isna().mean():.4f} | "
        f"unique_non_null={s.nunique(dropna=True):,}"
    )

    mi_t0 = time.perf_counter()
    mi_score = mutual_info_single_feature(x=s, y=y_all, feature_type=feature_type, random_state=random_state)
    print(f"    MI done in {time.perf_counter() - mi_t0:.1f}s | mi_score={mi_score:.6f}")

    probe_stats = cross_validated_univariate_probe(
        x=s,
        y=y_all,
        feature_type=feature_type,
        n_splits=cv_splits_per_feature,
        random_state=random_state,
        spline_df=4,
        ridge_alpha=1.0,
        feature_name=col,
        verbose=True,
    )

    rows.append(
        {
            "feature": col,
            "feature_group": get_feature_group(col),
            "source_ablation_group": get_feature_group(col),
            "feature_type": feature_type,
            "missing_rate": float(s.isna().mean()),
            "n_unique_non_null": int(s.nunique(dropna=True)),
            "mi_score": mi_score,
            **probe_stats,
        }
    )

    print(
        f"    feature finished in {time.perf_counter() - feat_t0:.1f}s | "
        f"elapsed total {time.perf_counter() - total_t0:.1f}s"
    )
    print("-" * 80)

per_feature_results = pd.DataFrame(rows)

per_feature_results = per_feature_results.sort_values(
    by=["feature_group", "probe_cv_r2", "mi_score"], ascending=[True, False, False]
).reset_index(drop=True)

per_feature_results

Using 2,000,000 rows for per-feature MI/probe scoring.


C:\Users\zpz\AppData\Local\Temp\ipykernel_31880\3730604924.py:73: Pandas4Warning: is_categorical_dtype is deprecated and will be removed in a future version. Use isinstance(dtype, pd.CategoricalDtype) instead
  if pd.api.types.is_object_dtype(s) or pd.api.types.is_string_dtype(s) or pd.api.types.is_categorical_dtype(s):


[1/162] feature=building_id | type=categorical | missing=0.0000 | unique_non_null=1,425
    MI done in 15.4s | mi_score=1.629829
      probe finished for building_id in 2.2s | R2=0.265496 | RMSE=2121.416154
    feature finished in 17.9s | elapsed total 17.9s
--------------------------------------------------------------------------------


C:\Users\zpz\AppData\Local\Temp\ipykernel_31880\3730604924.py:73: Pandas4Warning: is_categorical_dtype is deprecated and will be removed in a future version. Use isinstance(dtype, pd.CategoricalDtype) instead
  if pd.api.types.is_object_dtype(s) or pd.api.types.is_string_dtype(s) or pd.api.types.is_categorical_dtype(s):


[2/162] feature=ts_idx | type=categorical | missing=0.0000 | unique_non_null=8,784
    MI done in 54.4s | mi_score=0.012063
      probe finished for ts_idx in 2.3s | R2=-0.005557 | RMSE=2482.174349
    feature finished in 56.9s | elapsed total 74.8s
--------------------------------------------------------------------------------


...

C:\Users\zpz\AppData\Local\Temp\ipykernel_31880\3730604924.py:73: Pandas4Warning: is_categorical_dtype is deprecated and will be removed in a future version. Use isinstance(dtype, pd.CategoricalDtype) instead
  if pd.api.types.is_object_dtype(s) or pd.api.types.is_string_dtype(s) or pd.api.types.is_categorical_dtype(s):


    MI done in 7.8s | mi_score=0.000000
      probe finished for site_id in 2.0s | R2=0.025531 | RMSE=2443.503121
    feature finished in 10.0s | elapsed total 643.4s
--------------------------------------------------------------------------------


C:\Users\zpz\AppData\Local\Temp\ipykernel_31880\3730604924.py:73: Pandas4Warning: is_categorical_dtype is deprecated and will be removed in a future version. Use isinstance(dtype, pd.CategoricalDtype) instead
  if pd.api.types.is_object_dtype(s) or pd.api.types.is_string_dtype(s) or pd.api.types.is_categorical_dtype(s):


[35/162] feature=meter | type=categorical | missing=0.0000 | unique_non_null=4
    MI done in 8.5s | mi_score=0.000000
      probe finished for meter in 1.8s | R2=0.034263 | RMSE=2432.530678
    feature finished in 10.5s | elapsed total 653.9s
--------------------------------------------------------------------------------
[36/162] feature=thermal_heat_stress | type=numeric | missing=0.0000 | unique_non_null=636,032
    MI done in 12.0s | mi_score=0.170150
      probe finished for thermal_heat_stress in 2.4s | R2=0.011318 | RMSE=2461.258621
    feature finished in 14.6s | elapsed total 668.4s
--------------------------------------------------------------------------------
[37/162] feature=thermal_cool_stress | type=numeric | missing=0.0000 | unique_non_null=413,953
    MI done in 12.0s | mi_score=0.168196
      probe finished for thermal_cool_stress in 2.4s | R2=0.003274 | RMSE=2471.250089
    feature finished in 14.6s | elapsed total 683.0s
--------------------------------------------

,feature,feature_group,source_ablation_group,feature_type,missing_rate,n_unique_non_null,mi_score,probe_cv_r2,probe_cv_rmse,probe_model,n_rows_used
0,pair_id,base,base,categorical,0.0,2329,2.067205,0.591166,1582.713967,categorical_factor_ridge_sparse,2000000
1,building_id,base,base,categorical,0.0,1425,1.629829,0.265496,2121.416154,categorical_factor_ridge_sparse,2000000
2,log_sqft,base,base,numeric,0.0,1376,1.965561,0.043847,2420.430334,natural_cubic_spline_df4,2000000
3,meter,base,base,categorical,0.0,4,0.000000,0.034263,2432.530678,categorical_factor_ridge_sparse,2000000
4,site_id,base,base,categorical,0.0,16,0.000000,0.025531,2443.503121,categorical_factor_ridge_sparse,2000000
...,...,...,...,...,...,...,...,...,...,...,...
157,r5_local_hour,rank5,rank5,numeric,0.0,24,0.135001,0.000146,2475.125249,natural_cubic_spline_df4,2000000
158,r5_frac_primary_use_dayofyear,rank5,rank5,numeric,0.0,5330,0.200675,0.000115,2475.162924,natural_cubic_spline_df4,2000000
159,r5_is_day_off_or_holiday,rank5,rank5,boolean,0.0,2,0.000000,0.000052,2475.241200,boolean_factor_ridge_sparse,2000000
160,r5_is_weekend_local,rank5,rank5,boolean,0.0,2,0.000000,0.000051,2475.243018,boolean_factor_ridge_sparse,2000000


In [14]:
per_feature_results.to_csv("per_feature_quality_dependency_results.csv", index=False)

## Whole-set probe for baseline and baseline + each supplementary feature group

Old two-group setup is retained in the following code cell as comments; the revised version evaluates baseline plus expert, rank2, rank4, rank5, and rank25.

In [15]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

In [16]:
def _make_sparse_onehot_encoder(min_frequency=None, dtype=np.float32):
    kwargs = {
        "handle_unknown": "ignore",
        "dtype": dtype,
    }
    if min_frequency is not None:
        kwargs["min_frequency"] = min_frequency

    try:
        return OneHotEncoder(sparse_output=True, **kwargs)
    except TypeError:
        return OneHotEncoder(sparse=True, **kwargs)


def prepare_feature_set_frame(df_in: pd.DataFrame, feature_cols):
    """
    Prepare a mixed-type feature frame for the whole-set probe.
    - numeric -> numeric with NaN preserved for later median imputation
    - boolean/categorical -> string categories with '__MISSING__' filled in
    """
    X = df_in.loc[:, feature_cols].copy()

    type_map = {}
    numeric_cols = []
    discrete_cols = []

    for col in feature_cols:
        feature_type = infer_feature_type(X[col])
        type_map[col] = feature_type

        if feature_type == "numeric":
            X[col] = pd.to_numeric(X[col], errors="coerce")
            numeric_cols.append(col)

        elif feature_type == "boolean":
            X[col] = normalize_boolean_series(X[col]).astype("object").where(X[col].notna(), "__MISSING__").astype(str)
            discrete_cols.append(col)

        else:  # categorical
            X[col] = X[col].astype("object").where(X[col].notna(), "__MISSING__").astype(str)
            discrete_cols.append(col)

    return X, type_map, numeric_cols, discrete_cols


def make_feature_set_preprocessor(numeric_cols, discrete_cols, min_frequency=None):
    """
    Whole-set preprocessing:
    - numeric: median imputation
    - boolean/categorical: one-hot encoding
    """
    transformers = []

    if len(numeric_cols) > 0:
        transformers.append(
            (
                "num",
                Pipeline(
                    steps=[
                        ("imputer", SimpleImputer(strategy="median")),
                    ]
                ),
                numeric_cols,
            )
        )

    if len(discrete_cols) > 0:
        transformers.append(
            (
                "disc",
                _make_sparse_onehot_encoder(min_frequency=min_frequency),
                discrete_cols,
            )
        )

    return ColumnTransformer(
        transformers=transformers,
        remainder="drop",
        sparse_threshold=1.0,
    )

In [17]:
def evaluate_feature_set_probe(
    feature_cols,
    target_col,
    sample_size=None,
    n_splits=3,
    random_state=42,
    ridge_alpha=1.0,
    min_frequency=None,
):
    """
    Evaluate a whole feature set against the numeric target using a weak multivariate probe.

    Parameters
    ----------
    feature_cols : list[str]
        Feature columns to include.
    target_col : str
        Numeric target column.
    sample_size : int or None
        Optional row subsample for speed. If None, use all valid rows.
    n_splits : int
        CV folds. 3 is a good default for very large data.
    ridge_alpha : float
        Ridge regularization strength.
    min_frequency : int/float/None
        Optional OneHotEncoder infrequent-category grouping threshold.
        Leave as None unless cardinality becomes a runtime/memory issue.

    Returns
    -------
    dict
    """
    y_all = pd.to_numeric(df_full[target_col], errors="coerce")
    valid_mask = y_all.notna()

    X_raw = df_full.loc[valid_mask, feature_cols]
    y = y_all.loc[valid_mask]
    print(X_raw.head(5))

    # Optional subsample for speed
    if sample_size is not None and len(X_raw) > sample_size:
        sampled_idx = X_raw.sample(n=sample_size, random_state=random_state).index
        X_raw = X_raw.loc[sampled_idx]
        y = y.loc[sampled_idx]

    X_raw = X_raw.reset_index(drop=True)
    y = y.reset_index(drop=True)

    if len(y) < 2:
        return {
            "set_name": None,
            "n_rows_used": int(len(y)),
            "n_input_features": int(len(feature_cols)),
            "n_numeric_features": np.nan,
            "n_discrete_features": np.nan,
            "set_cv_r2": np.nan,
            "set_cv_rmse": np.nan,
            "ridge_alpha": ridge_alpha,
            "n_splits": n_splits,
            "sample_size": sample_size,
        }

    X, type_map, numeric_cols, discrete_cols = prepare_feature_set_frame(X_raw, feature_cols)

    preprocessor = make_feature_set_preprocessor(
        numeric_cols=numeric_cols,
        discrete_cols=discrete_cols,
        min_frequency=min_frequency,
    )

    model = Pipeline(
        steps=[
            ("prep", preprocessor),
            ("reg", Ridge(alpha=ridge_alpha, solver="lsqr")),
        ]
    )

    n_splits_eff = min(n_splits, len(y))
    if n_splits_eff < 2:
        return {
            "set_name": None,
            "n_rows_used": int(len(y)),
            "n_input_features": int(len(feature_cols)),
            "n_numeric_features": int(len(numeric_cols)),
            "n_discrete_features": int(len(discrete_cols)),
            "set_cv_r2": np.nan,
            "set_cv_rmse": np.nan,
            "ridge_alpha": ridge_alpha,
            "n_splits": n_splits_eff,
            "sample_size": sample_size,
        }

    cv = KFold(n_splits=n_splits_eff, shuffle=True, random_state=random_state)
    oof_pred = np.full(len(y), np.nan, dtype=float)

    for train_idx, test_idx in cv.split(X):
        X_train = X.iloc[train_idx]
        X_test = X.iloc[test_idx]
        y_train = y.iloc[train_idx]

        model.fit(X_train, y_train)
        oof_pred[test_idx] = model.predict(X_test)

    set_cv_r2 = r2_score(y, oof_pred)
    set_cv_rmse = np.sqrt(mean_squared_error(y, oof_pred))

    return {
        "set_name": None,
        "n_rows_used": int(len(y)),
        "n_input_features": int(len(feature_cols)),
        "n_numeric_features": int(len(numeric_cols)),
        "n_discrete_features": int(len(discrete_cols)),
        "set_cv_r2": float(set_cv_r2),
        "set_cv_rmse": float(set_cv_rmse),
        "ridge_alpha": ridge_alpha,
        "n_splits": n_splits_eff,
        "sample_size": sample_size,
    }

In [18]:
# Old code retained for reference.
# # You can start smaller for speed, e.g. 500_000 or 1_000_000
# sample_size_for_set_probe = 2_000_000
#
# base_set_stats = evaluate_feature_set_probe(
#     feature_cols=cols_base,
#     target_col=target_col,
#     sample_size=sample_size_for_set_probe,
#     n_splits=3,
#     random_state=42,
#     ridge_alpha=1.0,
#     min_frequency=None,
# )
# base_set_stats["set_name"] = "base_only"
#
# full_feature_cols = list(cols_base) + list(cols_exp)
# full_set_stats = evaluate_feature_set_probe(
#     feature_cols=full_feature_cols,
#     target_col=target_col,
#     sample_size=sample_size_for_set_probe,
#     n_splits=3,
#     random_state=42,
#     ridge_alpha=1.0,
#     min_frequency=None,
# )
# full_set_stats["set_name"] = "base_plus_exp"
#
# feature_set_results = pd.DataFrame([base_set_stats, full_set_stats])
#
# base_r2 = feature_set_results.loc[feature_set_results["set_name"] == "base_only", "set_cv_r2"].iloc[0]
# base_rmse = feature_set_results.loc[feature_set_results["set_name"] == "base_only", "set_cv_rmse"].iloc[0]
#
# feature_set_results["delta_r2_vs_base"] = feature_set_results["set_cv_r2"] - base_r2
# feature_set_results["delta_rmse_vs_base"] = feature_set_results["set_cv_rmse"] - base_rmse
#
# feature_set_results

# Revised: evaluate baseline and baseline + each supplementary feature group.
sample_size_for_set_probe = 2_000_000

probe_group_defs = [
    ("base_only", cols_base),
    ("base_plus_expert", list(cols_base) + list(cols_exp)),
    ("base_plus_rank2", list(cols_base) + list(cols_rank2)),
    ("base_plus_rank4", list(cols_base) + list(cols_rank4)),
    ("base_plus_rank5", list(cols_base) + list(cols_rank5)),
    ("base_plus_rank25", list(cols_base) + list(cols_rank25)),
]

feature_set_rows = []
for set_name, feature_cols in probe_group_defs:
    stats = evaluate_feature_set_probe(
        feature_cols=feature_cols,
        target_col=target_col,
        sample_size=sample_size_for_set_probe,
        n_splits=3,
        random_state=42,
        ridge_alpha=1.0,
        min_frequency=None,
    )
    stats["set_name"] = set_name
    feature_set_rows.append(stats)

feature_set_results = pd.DataFrame(feature_set_rows)

base_r2 = feature_set_results.loc[feature_set_results["set_name"] == "base_only", "set_cv_r2"].iloc[0]
base_rmse = feature_set_results.loc[feature_set_results["set_name"] == "base_only", "set_cv_rmse"].iloc[0]

feature_set_results["delta_r2_vs_base"] = feature_set_results["set_cv_r2"] - base_r2
feature_set_results["delta_rmse_vs_base"] = feature_set_results["set_cv_rmse"] - base_rmse

feature_set_results

  building_id ts_idx pair_id primary_use  square_feet  year_built  \
0           0   3384       0   Education         7432      2008.0   
1           1   3384       4   Education         2720      2004.0   
2           2   3384       8   Education         5376      1991.0   
3           3   3384      12   Education        23685      2002.0   
4           4   3384      16   Education       116607      1975.0   

   floor_count       timestamp_gmt  time_diff_hours  air_temperature  ...  \
0          NaN 2016-05-21 04:00:00              4.0             25.0  ...   
1          NaN 2016-05-21 04:00:00              4.0             25.0  ...   
2          NaN 2016-05-21 04:00:00              4.0             25.0  ...   
3          NaN 2016-05-21 04:00:00              4.0             25.0  ...   
4          NaN 2016-05-21 04:00:00              4.0             25.0  ...   

   is_business_hours  CDH_18C  HDH_18C  is_hot_24C  is_cold_10C  \
0                  0      7.0      0.0           1     

C:\Users\zpz\AppData\Local\Temp\ipykernel_31880\3730604924.py:73: Pandas4Warning: is_categorical_dtype is deprecated and will be removed in a future version. Use isinstance(dtype, pd.CategoricalDtype) instead
  if pd.api.types.is_object_dtype(s) or pd.api.types.is_string_dtype(s) or pd.api.types.is_categorical_dtype(s):
C:\Users\zpz\AppData\Local\Temp\ipykernel_31880\3730604924.py:73: Pandas4Warning: is_categorical_dtype is deprecated and will be removed in a future version. Use isinstance(dtype, pd.CategoricalDtype) instead
  if pd.api.types.is_object_dtype(s) or pd.api.types.is_string_dtype(s) or pd.api.types.is_categorical_dtype(s):
C:\Users\zpz\AppData\Local\Temp\ipykernel_31880\3730604924.py:73: Pandas4Warning: is_categorical_dtype is deprecated and will be removed in a future version. Use isinstance(dtype, pd.CategoricalDtype) instead
  if pd.api.types.is_object_dtype(s) or pd.api.types.is_string_dtype(s) or pd.api.types.is_categorical_dtype(s):
C:\Users\zpz\AppData\Local\Temp\ip

  building_id ts_idx pair_id primary_use  square_feet  year_built  \
0           0   3384       0   Education         7432      2008.0   
1           1   3384       4   Education         2720      2004.0   
2           2   3384       8   Education         5376      1991.0   
3           3   3384      12   Education        23685      2002.0   
4           4   3384      16   Education       116607      1975.0   

   floor_count       timestamp_gmt  time_diff_hours  air_temperature  ...  \
0          NaN 2016-05-21 04:00:00              4.0             25.0  ...   
1          NaN 2016-05-21 04:00:00              4.0             25.0  ...   
2          NaN 2016-05-21 04:00:00              4.0             25.0  ...   
3          NaN 2016-05-21 04:00:00              4.0             25.0  ...   
4          NaN 2016-05-21 04:00:00              4.0             25.0  ...   

   wx_latent_response  wx_envelope_response  meta_missing_score  \
0            2.859566              1.054335            

C:\Users\zpz\AppData\Local\Temp\ipykernel_31880\3730604924.py:73: Pandas4Warning: is_categorical_dtype is deprecated and will be removed in a future version. Use isinstance(dtype, pd.CategoricalDtype) instead
  if pd.api.types.is_object_dtype(s) or pd.api.types.is_string_dtype(s) or pd.api.types.is_categorical_dtype(s):
C:\Users\zpz\AppData\Local\Temp\ipykernel_31880\3730604924.py:73: Pandas4Warning: is_categorical_dtype is deprecated and will be removed in a future version. Use isinstance(dtype, pd.CategoricalDtype) instead
  if pd.api.types.is_object_dtype(s) or pd.api.types.is_string_dtype(s) or pd.api.types.is_categorical_dtype(s):
C:\Users\zpz\AppData\Local\Temp\ipykernel_31880\3730604924.py:73: Pandas4Warning: is_categorical_dtype is deprecated and will be removed in a future version. Use isinstance(dtype, pd.CategoricalDtype) instead
  if pd.api.types.is_object_dtype(s) or pd.api.types.is_string_dtype(s) or pd.api.types.is_categorical_dtype(s):
C:\Users\zpz\AppData\Local\Temp\ip

  building_id ts_idx pair_id primary_use  square_feet  year_built  \
0           0   3384       0   Education         7432      2008.0   
1           1   3384       4   Education         2720      2004.0   
2           2   3384       8   Education         5376      1991.0   
3           3   3384      12   Education        23685      2002.0   
4           4   3384      16   Education       116607      1975.0   

   floor_count       timestamp_gmt  time_diff_hours  air_temperature  ...  \
0          NaN 2016-05-21 04:00:00              4.0             25.0  ...   
1          NaN 2016-05-21 04:00:00              4.0             25.0  ...   
2          NaN 2016-05-21 04:00:00              4.0             25.0  ...   
3          NaN 2016-05-21 04:00:00              4.0             25.0  ...   
4          NaN 2016-05-21 04:00:00              4.0             25.0  ...   

   r2_wind_speed_site_z  r2_wind_speed_site_pos  r2_cloud_coverage_site_mean  \
0             -0.808319                0.0

...

  building_id ts_idx pair_id primary_use  square_feet  year_built  \
0           0   3384       0   Education         7432      2008.0   
1           1   3384       4   Education         2720      2004.0   
2           2   3384       8   Education         5376      1991.0   
3           3   3384      12   Education        23685      2002.0   
4           4   3384      16   Education       116607      1975.0   

   floor_count       timestamp_gmt  time_diff_hours  air_temperature  ...  \
0          NaN 2016-05-21 04:00:00              4.0             25.0  ...   
1          NaN 2016-05-21 04:00:00              4.0             25.0  ...   
2          NaN 2016-05-21 04:00:00              4.0             25.0  ...   
3          NaN 2016-05-21 04:00:00              4.0             25.0  ...   
4          NaN 2016-05-21 04:00:00              4.0             25.0  ...   

   site_id  meter  r5_local_hour  r5_local_weekday  r5_local_dayofyear  \
0        0      0             20                

C:\Users\zpz\AppData\Local\Temp\ipykernel_31880\3730604924.py:73: Pandas4Warning: is_categorical_dtype is deprecated and will be removed in a future version. Use isinstance(dtype, pd.CategoricalDtype) instead
  if pd.api.types.is_object_dtype(s) or pd.api.types.is_string_dtype(s) or pd.api.types.is_categorical_dtype(s):
C:\Users\zpz\AppData\Local\Temp\ipykernel_31880\3730604924.py:73: Pandas4Warning: is_categorical_dtype is deprecated and will be removed in a future version. Use isinstance(dtype, pd.CategoricalDtype) instead
  if pd.api.types.is_object_dtype(s) or pd.api.types.is_string_dtype(s) or pd.api.types.is_categorical_dtype(s):
C:\Users\zpz\AppData\Local\Temp\ipykernel_31880\3730604924.py:73: Pandas4Warning: is_categorical_dtype is deprecated and will be removed in a future version. Use isinstance(dtype, pd.CategoricalDtype) instead
  if pd.api.types.is_object_dtype(s) or pd.api.types.is_string_dtype(s) or pd.api.types.is_categorical_dtype(s):
C:\Users\zpz\AppData\Local\Temp\ip

  building_id ts_idx pair_id primary_use  square_feet  year_built  \
0           0   3384       0   Education         7432      2008.0   
1           1   3384       4   Education         2720      2004.0   
2           2   3384       8   Education         5376      1991.0   
3           3   3384      12   Education        23685      2002.0   
4           4   3384      16   Education       116607      1975.0   

   floor_count       timestamp_gmt  time_diff_hours  air_temperature  ...  \
0          NaN 2016-05-21 04:00:00              4.0             25.0  ...   
1          NaN 2016-05-21 04:00:00              4.0             25.0  ...   
2          NaN 2016-05-21 04:00:00              4.0             25.0  ...   
3          NaN 2016-05-21 04:00:00              4.0             25.0  ...   
4          NaN 2016-05-21 04:00:00              4.0             25.0  ...   

   r25_site_missing_rate_precip_depth_1_hr  r25_is_missing_sea_level_pressure  \
0                                      0.

C:\Users\zpz\AppData\Local\Temp\ipykernel_31880\3730604924.py:73: Pandas4Warning: is_categorical_dtype is deprecated and will be removed in a future version. Use isinstance(dtype, pd.CategoricalDtype) instead
  if pd.api.types.is_object_dtype(s) or pd.api.types.is_string_dtype(s) or pd.api.types.is_categorical_dtype(s):
C:\Users\zpz\AppData\Local\Temp\ipykernel_31880\3730604924.py:73: Pandas4Warning: is_categorical_dtype is deprecated and will be removed in a future version. Use isinstance(dtype, pd.CategoricalDtype) instead
  if pd.api.types.is_object_dtype(s) or pd.api.types.is_string_dtype(s) or pd.api.types.is_categorical_dtype(s):
C:\Users\zpz\AppData\Local\Temp\ipykernel_31880\3730604924.py:73: Pandas4Warning: is_categorical_dtype is deprecated and will be removed in a future version. Use isinstance(dtype, pd.CategoricalDtype) instead
  if pd.api.types.is_object_dtype(s) or pd.api.types.is_string_dtype(s) or pd.api.types.is_categorical_dtype(s):
C:\Users\zpz\AppData\Local\Temp\ip

,set_name,n_rows_used,n_input_features,n_numeric_features,n_discrete_features,set_cv_r2,set_cv_rmse,ridge_alpha,n_splits,sample_size,delta_r2_vs_base,delta_rmse_vs_base
0,base_only,2000000,35,21,14,0.050326,2412.215022,1.0,3,2000000,0.000000e+00,0.000000e+00
1,base_plus_expert,2000000,51,37,14,0.050326,2412.215022,1.0,3,2000000,9.804602e-12,-1.245235e-08
2,base_plus_rank2,2000000,59,45,14,0.050326,2412.215022,1.0,3,2000000,4.630141e-11,-5.880383e-08
3,base_plus_rank4,2000000,97,83,14,0.054520,2406.883841,1.0,3,2000000,4.193065e-03,-5.331181e+00
4,base_plus_rank5,2000000,43,27,16,0.050326,2412.215022,1.0,3,2000000,2.220013e-11,-2.819479e-08
5,base_plus_rank25,2000000,52,22,30,0.050326,2412.215022,1.0,3,2000000,5.773160e-15,-7.275958e-12


In [19]:
for _, row in feature_set_results.iterrows():
    print(
        f"{row['set_name']}: "
        f"R2={row['set_cv_r2']:.6f}, "
        f"RMSE={row['set_cv_rmse']:.6f}, "
        f"ΔR2_vs_base={row['delta_r2_vs_base']:.6f}, "
        f"ΔRMSE_vs_base={row['delta_rmse_vs_base']:.6f}, "
        f"rows={row['n_rows_used']}, "
        f"input_features={row['n_input_features']}"
    )

base_only: R2=0.050326, RMSE=2412.215022, ΔR2_vs_base=0.000000, ΔRMSE_vs_base=0.000000, rows=2000000, input_features=35
base_plus_expert: R2=0.050326, RMSE=2412.215022, ΔR2_vs_base=0.000000, ΔRMSE_vs_base=-0.000000, rows=2000000, input_features=51
base_plus_rank2: R2=0.050326, RMSE=2412.215022, ΔR2_vs_base=0.000000, ΔRMSE_vs_base=-0.000000, rows=2000000, input_features=59
base_plus_rank4: R2=0.054520, RMSE=2406.883841, ΔR2_vs_base=0.004193, ΔRMSE_vs_base=-5.331181, rows=2000000, input_features=97
base_plus_rank5: R2=0.050326, RMSE=2412.215022, ΔR2_vs_base=0.000000, ΔRMSE_vs_base=-0.000000, rows=2000000, input_features=43
base_plus_rank25: R2=0.050326, RMSE=2412.215022, ΔR2_vs_base=0.000000, ΔRMSE_vs_base=-0.000000, rows=2000000, input_features=52
